# Main experiment: substrate vs main + side chains

Same featurizer (ECFP count, radius 2, 1024 bits), same models and same 10 folds for every input; only the input changes:

| Experiment | Input to ECFP |
|---|---|
| Substrate | whole substrate SMILES |
| Main + Side | main-chain and side-chain ECFP concatenated (side all zeros for substrates without side chains) |
| Substrate + Main + Side | all three concatenated |

Results come from `scripts/models_training.py` with `configs/baseline_cof_matched.yaml` (Substrate) and `configs/split_cof_matched.yaml` (the rest).

In [1]:
import os

import joblib
import numpy as np
import pandas as pd

baseline_dir = "../results/baseline_cof_matched_rth_0.99/"
split_dir = "../results/split_cof_matched_rth_0.99/"

EXPERIMENTS = {
    "Substrate": (baseline_dir, "features_rth_0.99_ecfp_2_1024_True"),
    "Main + Side": (split_dir, "features_rth_0.99_main_ecfp_2_1024_True_side_ecfp_2_1024_True"),
    "Substrate + Main + Side": (split_dir, "features_rth_0.99_ecfp_2_1024_True_main_ecfp_2_1024_True_side_ecfp_2_1024_True"),
}

MODEL_DISPLAY_NAMES = {
    "bayes": "BRR",
    "knn": "KNN",
    "lasso": "LR",
    "mlp": "MLP",
    "random_forest": "RF",
    "xgboost": "XGB",
}

In [2]:
results = {"experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}

for experiment, (root_dir, dataset) in EXPERIMENTS.items():
    for model, model_name in MODEL_DISPLAY_NAMES.items():
        scores = joblib.load(os.path.join(root_dir, model, dataset, "scores.pickle"))
        results["experiment"].append(experiment)
        results["model"].append(model_name)
        results["smape_mean"].append(np.mean(scores["smape"]))
        results["smape_std"].append(np.std(scores["smape"]))
        results["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
        results["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))

df_results = pd.DataFrame(results)
df_results

,experiment,model,smape_mean,smape_std,pa_mean,pa_std
0,Substrate,BRR,0.609443,0.107758,0.687972,0.051538
1,Substrate,KNN,0.658913,0.112259,0.634068,0.089120
2,Substrate,LR,0.643621,0.080449,0.679068,0.079777
3,Substrate,MLP,0.652965,0.081939,0.672821,0.074619
4,Substrate,RF,0.658686,0.117282,0.650140,0.128908
5,Substrate,XGB,0.646526,0.117062,0.665734,0.136186
6,Main + Side,BRR,0.634389,0.085718,0.665641,0.077863
7,Main + Side,KNN,0.668314,0.108717,0.645769,0.093966
8,Main + Side,LR,0.714082,0.120256,0.671597,0.059659
9,Main + Side,MLP,0.690633,0.162330,0.625128,0.062045


In [3]:
def to_matrix(values: str) -> pd.DataFrame:
    matrix = df_results.pivot(index="experiment", columns="model", values=values)
    return matrix.loc[list(EXPERIMENTS), list(MODEL_DISPLAY_NAMES.values())]


smape_mean_matrix = to_matrix("smape_mean")
smape_std_matrix = to_matrix("smape_std")
pa_mean_matrix = to_matrix("pa_mean")
pa_std_matrix = to_matrix("pa_std")

smape_matrix = (smape_mean_matrix * 100).round(1).astype(str) + "(" + smape_std_matrix.round(2).astype(str) + ")"
pa_matrix = (pa_mean_matrix * 100).round(1).astype(str) + "(" + pa_std_matrix.round(2).astype(str) + ")"

In [4]:
smape_mean_matrix

model,BRR,KNN,LR,MLP,RF,XGB
experiment,,,,,,
Substrate,0.609443,0.658913,0.643621,0.652965,0.658686,0.646526
Main + Side,0.634389,0.668314,0.714082,0.690633,0.677021,0.656719
Substrate + Main + Side,0.623678,0.684352,0.670668,0.699814,0.680639,0.657732


In [5]:
pa_mean_matrix

model,BRR,KNN,LR,MLP,RF,XGB
experiment,,,,,,
Substrate,0.687972,0.634068,0.679068,0.672821,0.650140,0.665734
Main + Side,0.665641,0.645769,0.671597,0.625128,0.645664,0.654266
Substrate + Main + Side,0.667879,0.621632,0.661235,0.636270,0.658112,0.651585


## Same experiment with the previously best feature sets

Instead of ECFP alone, the featurizer is the feature set that worked best in the greedy forward selection (`greedy_selection_analysis.ipynb`):

| Feature set | Features |
|---|---|
| RF best | EState + functional groups |
| BRR best | custom descriptor + ECFP + EState + functional groups |

Each feature set is computed for the whole substrate, for the main chain + side chains, and for all three together (`configs/best_feature_generation.yaml`), and both models are trained on every variant (`configs/best_cof_matched.yaml`).

In [6]:
best_dir = "../results/best_cof_matched_rth_0.99/"

BEST_FEATURE_SETS = {"RF best": "rf_best", "BRR best": "brr_best"}
BEST_EXPERIMENTS = {"Substrate": "substrate", "Main + Side": "main_side", "Substrate + Main + Side": "substrate_main_side"}
BEST_MODELS = {"random_forest": "RF", "bayes": "BRR"}

results_best = {"feature_set": [], "experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}

for feature_set, feature_set_name in BEST_FEATURE_SETS.items():
    for experiment, experiment_name in BEST_EXPERIMENTS.items():
        for model, model_name in BEST_MODELS.items():
            scores = joblib.load(os.path.join(best_dir, model, f"features_rth_0.99_{feature_set_name}_{experiment_name}", "scores.pickle"))
            results_best["feature_set"].append(feature_set)
            results_best["experiment"].append(experiment)
            results_best["model"].append(model_name)
            results_best["smape_mean"].append(np.mean(scores["smape"]))
            results_best["smape_std"].append(np.std(scores["smape"]))
            results_best["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
            results_best["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))

df_results_best = pd.DataFrame(results_best)
df_results_best

,feature_set,experiment,model,smape_mean,smape_std,pa_mean,pa_std
0,RF best,Substrate,RF,0.600064,0.105020,0.688485,0.123328
1,RF best,Substrate,BRR,0.659057,0.062595,0.661212,0.060375
2,RF best,Main + Side,RF,0.642620,0.094280,0.648415,0.102090
3,RF best,Main + Side,BRR,0.649962,0.067336,0.643112,0.093155
4,RF best,Substrate + Main + Side,RF,0.661177,0.177806,0.653683,0.157248
5,RF best,Substrate + Main + Side,BRR,0.653001,0.066165,0.653380,0.078215
6,BRR best,Substrate,RF,0.594628,0.131369,0.663916,0.114507
7,BRR best,Substrate,BRR,0.614371,0.097749,0.701282,0.049793
8,BRR best,Main + Side,RF,0.611496,0.103151,0.684942,0.124431
9,BRR best,Main + Side,BRR,0.630965,0.081030,0.673193,0.082164


In [7]:
def to_best_matrix(values: str) -> pd.DataFrame:
    matrix = df_results_best.pivot(index=["feature_set", "experiment"], columns="model", values=values)
    rows = [(feature_set, experiment) for feature_set in BEST_FEATURE_SETS for experiment in BEST_EXPERIMENTS]
    return matrix.loc[rows, list(BEST_MODELS.values())]


smape_mean_matrix_best = to_best_matrix("smape_mean")
pa_mean_matrix_best = to_best_matrix("pa_mean")

In [8]:
smape_mean_matrix_best

model                                      RF       BRR
feature_set experiment                                 
RF best     Substrate                0.600064  0.659057
            Main + Side              0.642620  0.649962
            Substrate + Main + Side  0.661177  0.653001
BRR best    Substrate                0.594628  0.614371
            Main + Side              0.611496  0.630965
            Substrate + Main + Side  0.597003  0.623571

In [9]:
pa_mean_matrix_best

model                                      RF       BRR
feature_set experiment                                 
RF best     Substrate                0.688485  0.661212
            Main + Side              0.648415  0.643112
            Substrate + Main + Side  0.653683  0.653380
BRR best    Substrate                0.663916  0.701282
            Main + Side              0.684942  0.673193
            Substrate + Main + Side  0.682145  0.683310

## Same experiment evaluated on CTFs

The main/side split is most reliable for the 29 CTF substrates (11 of them have side chains). Two protocols:

| Protocol | Training | Test | Script and config |
|---|---|---|---|
| CTFs only | CTFs | CTFs (3 folds x 4 repeats) | `scripts/models_training.py`, `configs/best_ctf_only.yaml` |
| CTF test | half of the CTFs + all other COFs | the other half of the CTFs (2 folds x 5 repeats) | `scripts/models_training_cft_cof.py`, `configs/best_ctf_test.yaml` |

SMAPE of predicting the training median: 0.396 (CTFs only) and 0.441 (CTF test). These numbers are not comparable with the tables above, which are evaluated on all substrates.

In [10]:
ctf_only_dir = "../results/best_ctf_only/"
ctf_test_dir = "../results/best_ctf_test/"

CTF_FEATURE_SETS = {"ECFP": "ecfp", **BEST_FEATURE_SETS}


def load_ctf_results(root_dir: str) -> pd.DataFrame:
    results_ctf = {"feature_set": [], "experiment": [], "model": [], "smape_mean": [], "smape_std": [], "pa_mean": [], "pa_std": []}
    for feature_set, feature_set_name in CTF_FEATURE_SETS.items():
        for experiment, experiment_name in BEST_EXPERIMENTS.items():
            for model, model_name in BEST_MODELS.items():
                scores = joblib.load(os.path.join(root_dir, model, f"features_rth_0.99_{feature_set_name}_{experiment_name}", "scores.pickle"))
                results_ctf["feature_set"].append(feature_set)
                results_ctf["experiment"].append(experiment)
                results_ctf["model"].append(model_name)
                results_ctf["smape_mean"].append(np.mean(scores["smape"]))
                results_ctf["smape_std"].append(np.std(scores["smape"]))
                results_ctf["pa_mean"].append(np.mean(scores["pairwise_accuracy_score"]))
                results_ctf["pa_std"].append(np.std(scores["pairwise_accuracy_score"]))
    return pd.DataFrame(results_ctf)


def to_ctf_matrix(df: pd.DataFrame, values: str) -> pd.DataFrame:
    matrix = df.pivot(index=["feature_set", "experiment"], columns="model", values=values)
    rows = [(feature_set, experiment) for feature_set in CTF_FEATURE_SETS for experiment in BEST_EXPERIMENTS]
    return matrix.loc[rows, list(BEST_MODELS.values())]


df_results_ctf_only = load_ctf_results(ctf_only_dir)
df_results_ctf_test = load_ctf_results(ctf_test_dir)

### CTFs only: SMAPE and PA

In [11]:
to_ctf_matrix(df_results_ctf_only, "smape_mean")

model                                      RF       BRR
feature_set experiment                                 
ECFP        Substrate                0.374569  0.401996
            Main + Side              0.366288  0.401086
            Substrate + Main + Side  0.390071  0.398316
RF best     Substrate                0.446352  0.415576
            Main + Side              0.429970  0.417562
            Substrate + Main + Side  0.423887  0.416099
BRR best    Substrate                0.417261  0.406189
            Main + Side              0.379801  0.412565
            Substrate + Main + Side  0.409421  0.403454

In [12]:
to_ctf_matrix(df_results_ctf_only, "pa_mean")

model                                      RF       BRR
feature_set experiment                                 
ECFP        Substrate                0.607449  0.563721
            Main + Side              0.558102  0.562247
            Substrate + Main + Side  0.576873  0.560522
RF best     Substrate                0.470960  0.564015
            Main + Side              0.514962  0.553451
            Substrate + Main + Side  0.480682  0.570034
BRR best    Substrate                0.529693  0.572433
            Main + Side              0.568182  0.572012
            Substrate + Main + Side  0.504861  0.574369

### CTF test (other COFs in training): SMAPE and PA

In [13]:
to_ctf_matrix(df_results_ctf_test, "smape_mean")

model                                      RF       BRR
feature_set experiment                                 
ECFP        Substrate                0.417680  0.356077
            Main + Side              0.412488  0.371354
            Substrate + Main + Side  0.416258  0.356931
RF best     Substrate                0.471806  0.361111
            Main + Side              0.444223  0.371673
            Substrate + Main + Side  0.450985  0.366549
BRR best    Substrate                0.412215  0.349981
            Main + Side              0.422509  0.357208
            Substrate + Main + Side  0.401298  0.352378

In [14]:
to_ctf_matrix(df_results_ctf_test, "pa_mean")

model                                      RF       BRR
feature_set experiment                                 
ECFP        Substrate                0.616946  0.633120
            Main + Side              0.564853  0.606938
            Substrate + Main + Side  0.559361  0.619371
RF best     Substrate                0.507754  0.578698
            Main + Side              0.447750  0.619057
            Substrate + Main + Side  0.479268  0.611526
BRR best    Substrate                0.557311  0.642527
            Main + Side              0.564861  0.630388
            Substrate + Main + Side  0.543446  0.644298